# Ex.3: Attention from scratch and a tiny GPT
**EECE.4860/5860**: ungraded practice for Lectures 3–4.
1. Implement scaled dot-product attention with a causal mask and check it against PyTorch.
2. Train three tiny character-level GPTs of different sizes on Shakespeare and see how loss falls with model size, which is a small taste of scaling laws.

GPU time on a Colab T4: about 5 minutes. On a CPU, set `STEPS = 200`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/exercises/ex3_attention_tiny_gpt.ipynb)

In [ ]:
import importlib.util
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
import math, urllib.request
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from genai_utils import setup_env, measure
device, _ = setup_env()

## 1. Scaled dot-product attention
$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$, where the causal mask $M$ is $-\infty$ above the diagonal, so token $t$ only attends to tokens $\le t$ (Vaswani et al., 2017).

In [ ]:
def attention(q, k, v, causal=True):
    d_k = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if causal:
        T = q.shape[-2]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=q.device), diagonal=1)
        scores = scores.masked_fill(mask, float("-inf"))
    weights = scores.softmax(dim=-1)
    return weights @ v, weights

q, k, v = (torch.randn(2, 4, 5, 8) for _ in range(3))   # (batch, heads, tokens, d_k)
ours, w = attention(q, k, v)
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("matches PyTorch:", torch.allclose(ours, ref, atol=1e-5))
print("attention weights of head 0 (rows = query token):\n", w[0, 0].round(decimals=2))

## 2. A tiny GPT
Token + position embeddings, then `n_layer` blocks of (LayerNorm → causal multi-head attention → residual, LayerNorm → MLP → residual), then a linear layer to the vocabulary.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(URL).read().decode()
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
split = int(0.9 * len(data)); train_data, val_data = data[:split], data[split:]
BLOCK, VOCAB = 64, len(chars)
print(f"{len(text):,} characters, vocabulary {VOCAB}")

def batch(src, bs=64):
    ix = torch.randint(len(src) - BLOCK - 1, (bs,))
    x = torch.stack([src[i:i + BLOCK] for i in ix]); y = torch.stack([src[i + 1:i + BLOCK + 1] for i in ix])
    return x.to(device), y.to(device)

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.ln1, self.ln2 = h, nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(D, dim=-1)
        q, k, v = (t.view(B, T, self.h, D // self.h).transpose(1, 2) for t in (q, k, v))
        a, _ = attention(q, k, v)
        x = x + self.proj(a.transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, n_layer, d, h=4):
        super().__init__()
        self.tok, self.pos = nn.Embedding(VOCAB, d), nn.Embedding(BLOCK, d)
        self.blocks = nn.Sequential(*[Block(d, h) for _ in range(n_layer)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, VOCAB)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        return self.head(self.ln(self.blocks(x)))
    @torch.no_grad()
    def sample(self, n=300):
        idx = torch.zeros(1, 1, dtype=torch.long, device=device)
        for _ in range(n):
            p = self(idx[:, -BLOCK:])[:, -1].softmax(-1)
            idx = torch.cat([idx, torch.multinomial(p, 1)], dim=1)
        return "".join(chars[i] for i in idx[0].tolist())

## 3. Train three sizes with the same compute budget per model

In [ ]:
STEPS = 1500 if device == "cuda" else 200
results = []
for n_layer, d in ((1, 32), (2, 64), (4, 128)):
    torch.manual_seed(0)
    m = TinyGPT(n_layer, d).to(device)
    n_params = sum(p.numel() for p in m.parameters())
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    with measure(f"{n_layer} layers, d={d}"):
        for step in range(STEPS):
            x, y = batch(train_data)
            loss = F.cross_entropy(m(x).view(-1, VOCAB), y.view(-1))
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        val = sum(F.cross_entropy(m(x).view(-1, VOCAB), y.view(-1)).item() for x, y in (batch(val_data) for _ in range(20))) / 20
    results.append((n_params, val))
    print(f"params {n_params:,}: validation loss {val:.3f}")
print("\nSample from the largest model:\n", m.sample())

In [ ]:
plt.loglog([r[0] for r in results], [r[1] for r in results], "o-")
plt.xlabel("parameters"); plt.ylabel("validation loss"); plt.title("Bigger models reach lower loss (fixed steps)")
plt.show()

## 4. Try it
* Remove the causal mask (`causal=False`) and retrain the smallest model. Why does the training loss collapse while the samples get worse?
* Double `STEPS` for the smallest model. Does it catch up with the larger one? (Compare with the Chinchilla discussion in Lecture 4.)